This notebook makes plots and analyses of the homing/escape periods and exploration period using gaussian fitting to estimate the preferred tuning of each neuron.

Now that we have functions to get tuning curves for neurons using gaussian fits, next steps are:
- confirm that xval tuning curves are right, maybe change how that's done?
- compare tuning curve peak across conditions for each neuron
- are neurons tuned to the same fraction of the escape route?

And then we need to compare it to exploration:
- compute tuning curve for exploration
- what is overlap in which neurons are xval?
- are distance neurons tuned to the same thing in homind and explore
- if we subsample explore to the length of homing data, what are odds that we get homing data tuning?

Other things you need to implement:
- egocentric object vector tuning

In [1]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept, JAL3_22aug

from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_11thSept, JAL4_28aug

from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept, JAL005_5thSept

from behave_analysis.database.Experiments.JAL006_ex import JAL6_flip3_18mar, JAL6_flip7_1apr, JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar

from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr

from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip3_7may, JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may

#JAL3_7sept, JAL3_4sept, JAL3_1sept, JAL3_25aug, JAL3_22aug,
# 
experiments_objects = [JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept,
JAL005_8thSept, JAL005_21stSept, # JAL005_5thSept this one doesn't flip, but can be used as first barrier appearance
JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, # JAL6_flip7_1apr, # this session is sus
JAL7_sesh8_9apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_sesh9_16apr, JAL7_23apr,
JAL8_flip1_25apr,JAL8_flip2_29apr, JAL8_flip3_7may, JAL8_flip4_10may, JAL8_14may]

#
trials = [[1,3],[1,5,7],[1],[2],
    [2,3],[1,3],
    [1,3,4],[1,2,3],[1,2],[1,3],
    [4,5],[1,3],[3,4],[4,5],[2,5],
    [1,3],[1,2],[1,3],[5,7,8],[3,5],
]

In [82]:
%load_ext autoreload
from JR_test_scripts.escape.escape_utils import load, load_homing
from behave_analysis.utils.creating_directories import make_directory
from JR_test_scripts.escape.escape_data_loading_funcs import extract_homing_and_escape_periods, extract_explore_periods
from JR_test_scripts.escape.escape_plotting_funcs import plot_gaussian_fit_tuning, plot_pref_firing_condition, plot_tuning_matrix, tuning_curve_by_condition, tuning_curve_compare, xval_compare, plot_dist_pref_tuning_diff, plot_dist_pref_tuning_diff_compare, tuning_curve_sorter, pref_tuning_comparer_hist
from JR_test_scripts.escape.escape_tuning_funcs import neuron_tuning_by_var, single_trial_tuning

import numpy as np
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
"""Make plots for the neurons using their gaussian fitted tuning curves for sorting. Compare across conditions!
Plot only neurons with xval tuning curves"""

%autoreload 2
# compression_var = ['y_pos', 'distance_shelter', 'escape', 'speed'] 
compression_var = ['bird_dist_shelter','bird_dist_first_goal']

for i, exp in enumerate(experiments_objects[7:8]):
    # load data
    # session, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, escape, outofshelter = load(exp)
    # ons, offs, homie = load_homing(session, len(behave))
    for comp in compression_var:
        
        nickname = exp.nick_name + '_' + exp.experiment_date + '_' + comp
        dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/"+nickname)
        var, escape_matrix, cond, esc_start, h_start = extract_homing_and_escape_periods(session, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, comp, ons, offs, no_stationary = False, return_escape = True)
        
        # compute xval tuning during homing/escape
        peak_firing_condition, tuning, xval = neuron_tuning_by_var(var, escape_matrix, cond, h_start)
        mat_by_cond = single_trial_tuning(escape_matrix, var, cond, h_start)
        y_fitted, R, params, shift_constant, double_wins = plot_gaussian_fit_tuning(tuning, xval, dump_path, mat_by_cond, comp)
        pref_tuning_condition = np.zeros_like(xval)
        pref_tuning_condition[np.sum(xval, axis = 1)>0,:] = params[1,:,:] # for now only use the mu of the first gaussian regardless of double peak
        dump_path = "Z:/Jasmine_Laurence/summary_plots/all_trials/xval_gauss_tuning_compare" # saving folder name!
        plot_pref_firing_condition(peak_firing_condition = pref_tuning_condition, xval = xval, nickname = nickname + '_peak_firing', dump_path = dump_path)
        plot_dist_pref_tuning_diff(peak_firing_condition = pref_tuning_condition, xval = xval, nickname = nickname + '_peak_firing', dump_path = dump_path)
        tuning_curve_by_condition(tuning = tuning, xval = xval, nickname = nickname + '_tuning', peak_firing_condition = pref_tuning_condition, dump_path = dump_path)
        plot_tuning_matrix(tuning, cond, comp, escape_matrix, var, esc_start, h_start, xval, nickname = nickname + '_traces_sorted_tuning', peak_firing_condition = pref_tuning_condition, dump_path = dump_path)

In [102]:
"""
RELATIVE TO SHELTER vs RELATIVE TO GOAL AND SUBGOALS
Make plots for the neurons using their tuning curves for sorting. 
For distance and fraction of escape only, compare full route vs subcomponent of the route. 
Plot only neurons with xval tuning curves.
"""
%autoreload 2
compression_var = [['full_distance_shelter','distance_shelter','distance_first_goal'],['escape', 'escape_shelter', 'escape_first_goal']] # escape doesn't make sense here, obv
condition = ['shelter_only','barrier','flipped_barrier']

for i, exp in enumerate(experiments_objects[7:8]):
    # load data
    # session, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, escape, outofshelter = load(exp)
    # ons, offs, homie = load_homing(session, len(behave))
    for comp in compression_var:
        all_tuning = []
        all_pfc = []
        
        if isinstance(comp, list):
            for each_comp in comp:
                nickname = exp.nick_name + '_' + exp.experiment_date + '_' + each_comp
                dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/"+nickname)
                var, escape_matrix, cond, esc_start, h_start = extract_homing_and_escape_periods(session, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, each_comp, ons, offs, no_stationary = False, return_escape = True)
        
                # compute xval tuning during homing/escape
                peak_firing_condition, tuning, xval = neuron_tuning_by_var(var, escape_matrix, cond, h_start)
                mat_by_cond = single_trial_tuning(escape_matrix, var, cond, h_start)
                y_fitted, R, params, shift_constant, double_wins = plot_gaussian_fit_tuning(tuning, xval, dump_path, mat_by_cond, each_comp)
                pref_tuning_condition = np.zeros_like(xval)
                pref_tuning_condition[np.sum(xval, axis = 1)>0,:] = params[1,:,:] # for now only use the mu of the first gaussian regardless of double peak


                all_tuning.append(tuning)
                all_pfc.append(pref_tuning_condition) # if you do a gaussian fit for each compression var you can use that instead of the max firing

                if each_comp == comp[0]: # this saves our reference information - the full route in the shelter only condition
                    xval_sorter = xval
                    pfc = pref_tuning_condition
            
            # make a plot that takes in the pref tuning and takes the difference of pref tuning for each segment
            dump_path = "Z:/Jasmine_Laurence/summary_plots/all_trials/xval_gauss_tuning_compare" # saving folder name!
            nickname = exp.nick_name + '_' + exp.experiment_date

            if 'distance' in comp[0]:
                var_type = 'distance'
            if 'escape' in comp[0]:
                var_type = 'escape'
            tuning_curve_sorter(sorter = [all_tuning[0][0]], 
                                sortee = [all_tuning[0][1],all_tuning[1][1],all_tuning[2][1],all_tuning[0][2],all_tuning[1][2],all_tuning[2][2]], 
                                xval_sorter = [xval_sorter[:,0] == 1], 
                                name_sorter = [condition[0]+'\n'+comp[0]], 
                                name_sorted = [condition[1]+'\n'+comp[0],condition[1]+'\n'+comp[1],condition[1]+'\n'+comp[2],condition[2]+'\n'+comp[0],condition[2]+'\n'+comp[1],condition[2]+'\n'+comp[2]], 
                                nickname = nickname + '_tuning_' + var_type + '_compare', 
                                peak_firing_condition_sorter = [pfc[:,0]], 
                                dump_path = dump_path)
            pref_tuning_comparer_hist(reference = [pfc[:,0],pfc[:,0]], 
                                comparisons = [[all_pfc[0][:,1],all_pfc[1][:,1],all_pfc[2][:,1]],
                                                [all_pfc[0][:,2],all_pfc[1][:,2],all_pfc[2][:,2]]], 
                                xval_reference = [xval_sorter[:,0] == 1,xval_sorter[:,0] == 1], 
                                name_reference = [condition[0]+'\n'+comp[0], condition[0]+'\n'+comp[0]], 
                                name_comparisons = [[condition[1]+'\n'+comp[0],condition[1]+'\n'+comp[1],condition[1]+'\n'+comp[2]], 
                                                    [condition[2]+'\n'+comp[0],condition[2]+'\n'+comp[1],condition[2]+'\n'+comp[2]]], 
                                nickname = nickname + 'pref_tuning_' + var_type + '_hist_compare', 
                                dump_path = dump_path)

In [ ]:
"""
HOMING/ESCAPE vs EXPLORATION
Make plots for the neurons using their tuning curves for sorting. Compare to the tuning curves obtained from exploration periods. 
Plot only neurons with xval tuning curves.
TODO: subsample the exploration period to the length of the homing period?!
"""

%autoreload 2
# compression_var = ['y_pos', 'distance_shelter', 'speed', 'escape'] # escape doesn't make sense here, obv
compression_var = ['bird_dist_shelter']
for i, exp in enumerate(experiments_objects[7:8]):
    # load data
    # session, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, escape, outofshelter = load(exp)
    # ons, offs, homie = load_homing(session, len(behave))
    for comp in compression_var:
        
        nickname = exp.nick_name + '_' + exp.experiment_date + '_' + comp
        var, escape_matrix, cond, esc_start, h_start = extract_homing_and_escape_periods(session, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, comp, ons, offs, no_stationary = False, return_escape = True)
        if comp != 'escape':
            exp_var, explore_matrix, exp_cond = extract_explore_periods(session, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, comp, homie, escape, outofshelter, no_stationary = False)
        
        # compute xval tuning during homing/escape
        peak_firing_condition, tuning, xval = neuron_tuning_by_var(var, escape_matrix, cond, h_start)
        mat_by_cond = single_trial_tuning(escape_matrix, var, cond, h_start)
        dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/"+nickname)
        y_fitted, R, params, shift_constant, double_wins = plot_gaussian_fit_tuning(tuning, xval, dump_path, mat_by_cond, comp)
        pref_tuning_condition = np.zeros_like(xval)
        pref_tuning_condition[np.sum(xval, axis = 1)>0,:] = params[1,:,:] # for now only use the mu of the first gaussian regardless of double peak
        dump_path = "Z:/Jasmine_Laurence/summary_plots/all_trials/xval_gauss_tuning_compare" # saving folder name!
        plot_dist_pref_tuning_diff(peak_firing_condition = pref_tuning_condition, xval = xval, nickname = nickname + '_peak_firing_dist', dump_path = dump_path)
        
        if comp != 'escape':
            # compute xval tuning during explore
            exp_peak_firing_condition, exp_tuning, exp_xval = neuron_tuning_by_var(exp_var, explore_matrix, exp_cond, epoch_method = 'alt_time')
            mat_by_cond = []
            dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/" + nickname + '_explore')
            exp_y_fitted, exp_R, exp_params, exp_shift_constant, exp_double_wins = plot_gaussian_fit_tuning(exp_tuning, exp_xval, dump_path, mat_by_cond, comp)
            # plot the tuning during explore
            exp_pref_tuning_condition = np.zeros_like(exp_xval)
            exp_pref_tuning_condition[np.sum(exp_xval, axis = 1)>0,:] = exp_params[1,:,:]
            dump_path = "Z:/Jasmine_Laurence/summary_plots/all_trials/xval_gauss_tuning_compare" # saving folder name!
            plot_pref_firing_condition(peak_firing_condition = exp_pref_tuning_condition, xval = exp_xval, nickname = nickname + '_peak_firing_explore', dump_path = dump_path)
            plot_dist_pref_tuning_diff(peak_firing_condition = exp_pref_tuning_condition, xval = exp_xval, nickname = nickname + '_peak_firing_dist_explore', dump_path = dump_path)
            tuning_curve_by_condition(tuning = exp_tuning, xval = exp_xval, nickname = nickname + '_tuning_explore', peak_firing_condition = exp_pref_tuning_condition, dump_path = dump_path)
            # plot the comparison of tuning for explore and homing/excape
            plot_dist_pref_tuning_diff_compare(peak_firing_condition1 = pref_tuning_condition, 
                                               peak_firing_condition2 = exp_pref_tuning_condition, 
                                               xval1 = xval, 
                                               xval2 = exp_xval, 
                                               name1 = 'homing/escape', name2 = 'explore',
                                               nickname = nickname + '_peak_firing_compare', dump_path = dump_path)
            tuning_curve_compare(tuning = [tuning, exp_tuning], xval1 = xval, name = ['homing/escape', 'explore'], nickname = nickname + '_tuning_compare', peak_firing_condition = pref_tuning_condition, dump_path = dump_path)
            xval_compare(xval1 = xval, xval2 = exp_xval, name1 = 'homing/escape', name2 = 'explore', nickname = nickname + '_xval_compare', dump_path = dump_path)

TypeError: tuning_curve_compare() got an unexpected keyword argument 'tuning1'